# Store Data - Simple Baseline

- [Forecasting with ML and Stuff](https://www.kaggle.com/code/ryanholbrook/forecasting-with-machine-learning/tutorial)
- [Kaggle Page](https://www.kaggle.com/competitions/store-sales-time-series-forecasting/data)


## Imports

In [ ]:
import math
from collections import OrderedDict
from collections.abc import Iterable

import torch
from torch import Tensor, nn
from torch.utils.data import DataLoader, Subset

from common.modules import GetLastIndex, PositionalEncoding
from time_series.store_sales import MSLELoss, StoreData, Trainer

In [ ]:
device = (
    torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
)
device

# Notes

From the information
- Wages in the public sector are paid every two weeks on the 15 th and on the last day of the month. Supermarket sales could be affected by this.
- A magnitude 7.8 earthquake struck Ecuador on April 16, 2016. People rallied in relief efforts donating water and other first need products which greatly affected supermarket sales for several weeks after the earthquake.

# Load and Explore Data

In [ ]:
store_data = StoreData(dtype=torch.float32)
store_data_loader = DataLoader(store_data, batch_size=10)
store_data.train

# Model Design

- Input data is essentially `date x store_nbr x family`, which is easily tensor-able. The overall data will be `batch, date_len, store_nbr, family`. 
  - Remember to use `batch_first=True` for any LSTM or whatever models. Apparently having the batch dimension be second is an old convention.
- We need to predict 2017-08-16 to 2017-08-31, so 16 day prediction window
- Input window length will be a parameter, sent to the dataset and model
- loss is Root-Mean-Squared-Logarithmic-Error (given by problem)
  - $\text{RMSE}(\log(1+\hat{y}), \log(1+y))$

In [ ]:
class StoreSalesEncoderOnly(nn.Module):
    def __init__(
        self,
        input_embedding_shape: Iterable[int] = store_data.sales_tensor.shape[-2:],
        output_lags: int = store_data.output_lags,
        d_model: int = 128,
        num_layers: int = 6,
        nhead: int = 8,
        max_seq_length: int = 512,
    ) -> None:
        super().__init__()
        self.input_embedding_shape = input_embedding_shape
        self.output_lags = output_lags
        input_embedding_size = math.prod(input_embedding_shape)
        models = OrderedDict[str, nn.Module](
            [
                ("input_flatten", nn.Flatten(-2)),
                ("input_lin", nn.Linear(input_embedding_size, d_model)),
                ("pos_enc", PositionalEncoding(d_model, max_length=max_seq_length)),
                (
                    "encoder",
                    nn.TransformerEncoder(
                        nn.TransformerEncoderLayer(d_model, nhead, batch_first=True),
                        num_layers=num_layers,
                    ),
                ),
                # Pooling is just grab all outputs
                # ("flatten", nn.Flatten(-2)),
                ("last_index", GetLastIndex()),
                ("output_lin", nn.LazyLinear(input_embedding_size * output_lags)),
                (
                    "output_unflatten",
                    nn.Unflatten(-1, (self.output_lags, *self.input_embedding_shape)),
                ),
            ]
        )
        self.sequence = nn.Sequential(models)

    def forward(self, input_sequence: Tensor) -> Tensor:
        return self.sequence(input_sequence)


model = StoreSalesEncoderOnly()
model

# Training

In [ ]:
epochs = 500
save_every_n_epochs = 50
learning_rate = 1e-3
split_fraction = 0.9
batch_size = 128

split_loc = int(len(store_data) * split_fraction)

train_data = Subset(store_data, range(0, split_loc))
val_data = Subset(store_data, range(split_loc, len(store_data)))

train_loader = DataLoader[Tensor](train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader[Tensor](val_data, batch_size=batch_size)

In [ ]:
train_loader = DataLoader[Tensor](train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader[Tensor](val_data, batch_size=batch_size)
trainer = Trainer(
    model=model,
    device=device,
    train_loader=train_loader,
    val_loader=val_loader,
    learning_rate=learning_rate,
    loss_func=MSLELoss(),
)
trainer.train(epochs, save_every_n_epochs=save_every_n_epochs)

# Scrap

In [ ]:
# # Batch Size Sweep

# with mlflow.start_run(
#     run_name="batch_size_sweep",
#     tags={
#         "architecture": "transformer",
#         "input_window_days": str(store_data.window_lags),
#         "output_window_days": str(store_data.output_lags),
#         "device": str(device),
#         "git_branch": get_branch(),
#         "git_sha": get_sha(),
#     },
# ) as mlflow_run:
#     mlflow.log_params(
#         {
#             "epochs": epochs,
#             "learning_rate": learning_rate,
#             "split_fraction": split_fraction,
#             # "batch_size": batch_size,
#         }
#     )
#     for batch_size in [64, 128, 256, 512]:
#         with mlflow.start_run(nested=True) as subrun:
#             mlflow.log_param("batch_size", batch_size)
#             train_loader = DataLoader[Tensor](
#                 train_data, batch_size=batch_size, shuffle=True
#             )
#             val_loader = DataLoader[Tensor](val_data, batch_size=batch_size)
#             trainer = Trainer(model, train_loader=train_loader, val_loader=val_loader)
#             results_df = trainer.train(epochs)
#             results_df.plot(sharex=True, subplots=True, figsize=(8, 12))